# Reference Lab — your DNG, three reference looks
### P1060804.dng · personalized Colab notebook
Your original **DNG and all three reference JPEGs are embedded in this notebook**. No web image downloads, Google Drive setup or manual uploads are needed. Open in Colab and **Run all**.

The comparison applies each reference independently to the same RAW development: **teal/warm motion**, **muted olive/warm neutrals**, and **warm amber interior**. Embedded results below were rendered from your files, not the earlier Sony demo.

The workflow uses light and RGB curves across five tonal zones, optional region masks, adjustable controls and 16-bit TIFF export. It transfers tone and colour, not scene content, motion blur or new light sources. Your landscape and these references have different subjects, so global matching is the default; paired manual region masks remain available.

**Portability:** the notebook is larger because it includes the exact original DNG (~22 MiB). Colab still needs internet to install Python packages. The RAW is extracted unchanged and SHA-256 checked. Do not share this notebook publicly unless you intend to share the embedded photographs.


In [ ]:
# 1 · Install. Run once in a fresh Colab runtime.
%pip -q install "numpy>=1.26,<3" "scipy>=1.11,<2" "Pillow>=10,<13" "rawpy>=0.24,<0.29" "tifffile>=2024.1" "matplotlib>=3.8,<4" "ipywidgets>=8,<9" "psutil>=5.9" "pandas>=2,<3"
# Optional masks are installed only if enabled below.


In [ ]:
# 2 · Settings — safe defaults for ordinary Colab RAM
from pathlib import Path
import gc, io, json, time, warnings, importlib.metadata
import numpy as np
from PIL import Image, ImageOps, ImageCms
from scipy.ndimage import gaussian_filter
from scipy.interpolate import PchipInterpolator
import matplotlib.pyplot as plt
import psutil

SOURCE_PATH = ""  # e.g. /content/drive/MyDrive/photos/source.ARW
REFERENCE_PATH = ""
USE_DRIVE = False
USE_EMBEDDED_ASSETS = True
WEB_DEMO = False  # retained only for compatibility; no web assets are used
SELECTED_REFERENCE = 0          # 0 teal/warm, 1 olive/neutrals, 2 amber interior
AUTO_SUBJECT = False             # optional small u2netp salient-object model
PREVIEW_LONG_EDGE = 1280
RAW_HALF_SIZE = True            # preview decode: lowers RAW demosaic memory
EXPORT_LONG_EDGE = 4000         # 0 = native resolution; full RAW decode still needs RAM
EXPORT_FULL_RAW = True
TILE_ROWS = 128
# Optional white-on-black masks. They must match the ORIENTED images.
# Supply corresponding regions in BOTH images. "foreground" means near scenery,
# separate from the main subject; it is never guessed from vertical position.
SOURCE_MASKS = {"subject": "", "foreground": ""}
REFERENCE_MASKS = {"subject": "", "foreground": ""}
SEED = 41
np.random.seed(SEED)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
print(f"Available RAM: {psutil.virtual_memory().available / 2**30:.1f} GiB")


In [ ]:
EMBEDDED_ASSETS = [{'original_name': 'P1060804.dng', 'filename': 'source.dng', 'name': 'Your source DNG', 'sha256': '3e4c98e040fb150c6ea4723b53d46a5c7fc9e74930dc8194d37d8c686c73f8c3'}, {'original_name': 'Romantic Embrace in Motion.jpeg', 'filename': 'reference_1.jpg', 'name': 'Teal and warm motion', 'sha256': 'd7aef71c576985ae7f9bcf33a7aeebf1bd45330a43e38cd2f76b280214aef389'}, {'original_name': '_.jpeg', 'filename': 'reference_2.jpg', 'name': 'Muted olive and warm neutrals', 'sha256': '16ca794144bf7b1341231c0d07d22d871641769e369b6bf54c143ff00f1e0992'}, {'original_name': 'Warm Lighting Ideas That Make an Apartment Feel 10x Cozier.jpeg', 'filename': 'reference_3.jpg', 'name': 'Warm amber interior', 'sha256': '60b2edef76ffc96c8a67cf5fb4af2e00cb0f75e4a8c29c42c0de69d18fc08a5b'}]


In [ ]:
# 3 · Extract the attached originals, with checksum verification
import hashlib,base64
ASSET_DIR=Path('/content/reference_lab_attached') if Path('/content').exists() else Path('work/attached_assets')
ASSET_DIR.mkdir(parents=True,exist_ok=True)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
def extract_asset(a):
    path=Path('/kaggle/input/reference-lab-raw-assets')/a['filename']
    if not path.exists():
        matches=list(Path('/kaggle/input').rglob(a['filename']))
        if len(matches)!=1: raise FileNotFoundError(a['filename'])
        path=matches[0]
    if sha256_file(path)!=a['sha256']: raise ValueError('Checksum mismatch')
    print('Verified:',a['original_name'])
    return str(path)
def upload_one(label):
    from google.colab import files
    print(label,'— select one file')
    data=files.upload()
    if len(data)!=1:raise ValueError('Select exactly one file.')
    path=next(iter(data));del data;gc.collect();return path
if USE_EMBEDDED_ASSETS:
    assert 0<=SELECTED_REFERENCE<3
    paths=[extract_asset(a) for a in EMBEDDED_ASSETS]
    SOURCE_PATH=paths[0];REFERENCE_PATHS=paths[1:]
    REFERENCE_PATH=REFERENCE_PATHS[SELECTED_REFERENCE]
    REFERENCE_ASSETS=[{k:v for k,v in a.items() if k!='base64'} for a in EMBEDDED_ASSETS[1:]]
    SOURCE_MASKS={'subject':'','foreground':''};REFERENCE_MASKS={'subject':'','foreground':''}
else:
    if not SOURCE_PATH:SOURCE_PATH=upload_one('SOURCE RAW')
    if not REFERENCE_PATH:REFERENCE_PATH=upload_one('REFERENCE')
    REFERENCE_PATHS=[REFERENCE_PATH]
    REFERENCE_ASSETS=[{'name':Path(REFERENCE_PATH).stem}]
RAW_ASSET={k:v for k,v in EMBEDDED_ASSETS[0].items() if k!='base64'}
assert Path(SOURCE_PATH).is_file() and all(Path(p).is_file() for p in REFERENCE_PATHS)


### Your attached inputs
| Role | Original filename | Reference look |
|---|---|---|
| Source | `P1060804.dng` | Mountain landscape, decoded once for preview |
| Reference 1 | `Romantic Embrace in Motion.jpeg` | Teal shadows, warm highlights and deep blacks |
| Reference 2 | `_.jpeg` | Muted olive, warm skin/neutral colours |
| Reference 3 | `Warm Lighting Ideas That Make an Apartment Feel 10x Cozier.jpeg` | Strong amber/orange warmth |

Original bytes are embedded, extracted unchanged, and checked against their saved hashes. Reference 2 contains social-media icons and borders; the fitter uses its full image, so those pixels can slightly affect statistics. You can replace it with a clean crop when available. No references are sent to an external editing API.


## Develop and colour-manage
RAW uses camera white balance, no automatic brightness, 16-bit linear sRGB output and highlight reconstruction. Then apply the sRGB transfer function explicitly. Embedded profiles on ordinary reference images are converted to sRGB; untagged images are assumed sRGB. For reference inputs use an ICC-tagged JPEG/PNG. Pillow loads ordinary reference images at 8 bits; the RAW path retains higher precision.

Development exposure is separate from the later grading exposure. If the starting RAW is very dark, adjust `RAW_DEVELOP_EV` **before** mask extraction and fitting. The decoded RAW may already have clipped values; this pipeline cannot recover information the decoder has lost.


In [ ]:
# 4 · Core utilities
RAW_DEVELOP_EV = 0.0
RAW_EXT = {'.arw','.cr2','.cr3','.nef','.nrw','.raf','.rw2','.orf','.dng','.pef','.srw','.raw'}
def srgb_encode(x):
    x = np.maximum(x, 0)
    return np.where(x <= .0031308, 12.92*x, 1.055*np.power(x, 1/2.4)-.055).astype('float32')
def srgb_decode(x):
    x = np.maximum(x, 0)
    return np.where(x <= .04045, x/12.92, ((x+.055)/1.055)**2.4).astype('float32')
def resize_float(x, shape):
    h,w=shape
    if x.shape[:2] == (h,w): return x.astype('float32',copy=False)
    if x.ndim == 2:
        return np.asarray(Image.fromarray(x.astype('float32')).resize((w,h),Image.Resampling.BILINEAR)).copy()
    return np.stack([resize_float(x[...,i],shape) for i in range(x.shape[-1])],axis=-1)
def downsize(x, edge):
    h,w=x.shape[:2]
    scale=min(1,edge/max(h,w)) if edge else 1
    return resize_float(x,(max(1,round(h*scale)),max(1,round(w*scale))))
def light(x):
    # Display-referred luma, deliberately distinct from linear luminance.
    return x @ np.array([.2126,.7152,.0722],dtype='float32')
def load_photo(path, edge=1280, half=True):
    if Path(path).suffix.lower() in RAW_EXT:
        import rawpy
        with rawpy.imread(path) as raw:
            arr=raw.postprocess(use_camera_wb=True,no_auto_bright=True,
                output_color=rawpy.ColorSpace.sRGB,gamma=(1,1),output_bps=16,
                half_size=half,highlight_mode=rawpy.HighlightMode.ReconstructDefault)
        x=arr.astype('float32')/65535
        del arr
        x=srgb_encode(x * 2**RAW_DEVELOP_EV)
    else:
        with Image.open(path) as im:
            icc=im.info.get('icc_profile')
            im=ImageOps.exif_transpose(im)
            if icc:
                try:
                    im=ImageCms.profileToProfile(im,ImageCms.ImageCmsProfile(io.BytesIO(icc)),
                                                ImageCms.createProfile('sRGB'),outputMode='RGB')
                except Exception as e:
                    raise ValueError('ICC conversion failed; export reference as sRGB first.') from e
            else:
                print(Path(path).name, ': no ICC profile; assuming sRGB')
            x=np.asarray(im.convert('RGB'),dtype='float32')/255
    return np.clip(downsize(x,edge),0,1).astype('float32')
def show_images(images, titles, width=16):
    fig,axs=plt.subplots(1,len(images),figsize=(width,5),squeeze=False)
    for ax,x,t in zip(axs[0],images,titles):
        ax.imshow(np.clip(x,0,1),cmap='gray' if x.ndim==2 else None,vmin=0,vmax=1)
        ax.set_title(t); ax.axis('off')
    plt.tight_layout(); plt.show(); plt.close(fig)
source=load_photo(SOURCE_PATH,PREVIEW_LONG_EDGE,RAW_HALF_SIZE)
reference=load_photo(REFERENCE_PATH,PREVIEW_LONG_EDGE,True)
show_images([source,reference],['Source development','Reference look'])


## Region masks — inspect before accepting
The automatic model identifies a **salient subject**, not reliable semantic classes or depth. For landscapes, clutter, hair, glass and multiple subjects, supply masks yourself. A foreground mask is optional and must be supplied in both images. Anything not covered becomes background. Subject takes priority over foreground; masks sum to one.

For manual masks: export grayscale PNGs with white inside the region, black outside, matching image orientation and framing; upload them through Colab's Files panel and set their paths in cell 2. Source and reference masks can have different dimensions. **Match meaning, not screen position.** If the reference has no corresponding subject, disable automatic masks and use a global match, or choose a better reference.


In [ ]:
# 5 · Segmentation (one reusable CPU session; automatic failure is explicit)
_session=None
if AUTO_SUBJECT:
    import subprocess, sys
    subprocess.check_call([sys.executable,'-m','pip','-q','install','rembg[cpu]>=2.0.60,<3'])
    from rembg import remove,new_session
    _session=new_session('u2netp',providers=['CPUExecutionProvider'])
def read_mask(path,shape):
    with Image.open(path) as im:
        m=np.asarray(ImageOps.exif_transpose(im).convert('L'),dtype='float32')/255
    return np.clip(resize_float(m,shape),0,1)
def region_masks(x, paths):
    shape=x.shape[:2]
    if paths.get('subject'):
        subject=read_mask(paths['subject'],shape)
    elif AUTO_SUBJECT:
        small=downsize(x,768)
        im=Image.fromarray(np.uint8(np.clip(small,0,1)*255))
        m=remove(im,session=_session,only_mask=True)
        subject=resize_float(np.asarray(m.convert('L'),dtype='float32')/255,shape)
    else: subject=np.zeros(shape,dtype='float32')
    subject=np.clip(gaussian_filter(subject,1.0),0,1)
    foreground=read_mask(paths['foreground'],shape) if paths.get('foreground') else np.zeros(shape,dtype='float32')
    foreground=np.clip(gaussian_filter(foreground,1.0),0,1)*(1-subject)
    return {'subject':subject,'foreground':foreground,'background':np.clip(1-subject-foreground,0,1)}
sm=region_masks(source,SOURCE_MASKS)
rm=region_masks(reference,REFERENCE_MASKS)
for title,x,masks in [('Source',source,sm),('Reference',reference,rm)]:
    overlay=np.stack([masks['subject'],masks['foreground'],masks['background']],-1)
    show_images([x,.55*x+.45*overlay,*masks.values()],
                [title,'Red subject / green foreground / blue background',*masks.keys()])
print('Inspect edges and correspondence. Correct paths and rerun if needed.')


## Fit the look
Five broad tonal intervals are used for reporting: **0–20% blacks, 20–40% shadows, 40–60% midtones, 60–80% highlights, 80–100% whites**. Smooth overlapping weights centred at 0, .25, .5, .75, 1 drive adjustments, so boundaries never create bands.

For each valid paired region: fit a monotonic light curve from robust quantiles; then fit RGB residual curves and five-zone colour offsets. The RGB step removes its luma component to reduce conflicts with the light curve. Sparse zones receive reduced weight. Small/missing regions fall back to the global transform with a warning. Regularized curves and capped shifts protect against extreme changes, but are not a guarantee of natural skin.


In [ ]:
# 6 · Fit engine — bounded sampling, no neural training
GRID=np.linspace(0,1,1024,dtype='float32')
CENTERS=np.linspace(0,1,5,dtype='float32')
ZONE_NAMES=['Blacks','Shadows','Midtones','Highlights','Whites']
def zone_weights(y):
    z=np.maximum(1-np.abs(y[...,None]-CENTERS)/.25,0)
    return z/np.maximum(z.sum(-1,keepdims=True),1e-8)
def sample_pixels(x,mask,n=40000):
    idx=np.flatnonzero(mask.ravel()>.55)
    if len(idx)<64: return None
    rng=np.random.default_rng(SEED)
    if len(idx)>n: idx=rng.choice(idx,n,replace=False)
    return x.reshape(-1,3)[idx]
def fit_curve(a,b,max_shift=.25):
    qs=np.linspace(.01,.99,25)
    xa=np.quantile(a,qs); yb=np.quantile(b,qs)
    # Duplicate quantiles occur in flat/clipped areas; never pass them to PCHIP.
    ux,inv=np.unique(xa,return_inverse=True)
    uy=np.bincount(inv,weights=yb)/np.bincount(inv)
    if len(ux)<3 or np.ptp(a)<1e-4: return GRID.copy()
    inside=(ux>1e-5)&(ux<1-1e-5)
    xx=np.r_[0,ux[inside],1]; yy=np.r_[0,uy[inside],1]
    lut=PchipInterpolator(xx,np.maximum.accumulate(yy))(GRID)
    lut=np.clip(lut,GRID-max_shift,GRID+max_shift)
    return np.clip(.15*GRID+.85*lut,0,1).astype('float32')
def apply_light(x,lut):
    y=light(x); yt=np.interp(y,GRID,lut).astype('float32')
    # Additive luma displacement preserves chroma differences, avoids zero division.
    return x+(yt-y)[...,None]
def apply_model(x,model,tone=1,color=1):
    base=apply_light(x,model['light'])
    base=x+tone*(base-x)
    curved=np.stack([np.interp(np.clip(base[...,c],0,1),GRID,model['rgb'][c])
                     for c in range(3)],-1).astype('float32')
    delta=curved-base
    delta-=light(delta)[...,None]
    offsets=zone_weights(np.clip(light(base),0,1)) @ np.asarray(model['zones'],dtype='float32')
    return base+color*(.65*delta+.35*offsets)
def fit_model(a,b):
    lum=fit_curve(light(a),light(b),.30)
    at=apply_light(a,lum)
    curves=np.stack([fit_curve(np.clip(at[:,c],0,1),b[:,c],.18) for c in range(3)])
    za=zone_weights(np.clip(light(at),0,1)); zb=zone_weights(light(b))
    offsets=[]
    for k in range(5):
        wa,wb=za[:,k],zb[:,k]
        ma=(at*wa[:,None]).sum(0)/max(wa.sum(),1)
        mb=(b*wb[:,None]).sum(0)/max(wb.sum(),1)
        d=np.clip(mb-ma,-.15,.15); d-=light(d)
        reliability=min(1,wa.sum()/200,wb.sum()/200)
        offsets.append(d*reliability)
    return {'light':lum,'rgb':curves,'zones':np.asarray(offsets,dtype='float32')}
def fit_all(s,r,sm,rm):
    models={'global':fit_model(sample_pixels(s,np.ones(s.shape[:2])),sample_pixels(r,np.ones(r.shape[:2])))}
    valid=[]
    for name in sm:
        a,b=sample_pixels(s,sm[name]),sample_pixels(r,rm[name])
        if a is None or b is None or sm[name].mean()<.015 or rm[name].mean()<.015:
            if max(sm[name].mean(),rm[name].mean())>.015:
                warnings.warn(f'{name}: missing/too small in one image; using global grade.')
            continue
        models[name]=fit_model(a,b); valid.append(name)
    return models,valid
start=time.time()
models,valid_regions=fit_all(source,reference,sm,rm)
print('Fitted regions:',valid_regions,'in',round(time.time()-start,2),'seconds')


In [ ]:
# 7 · Inspect the requested 4 channels × 5 tonal zones
# Columns run WHITE → BLACK; plotted horizontal axes always run black → white.
import pandas as pd
def channel_table(x):
    y=light(x); band=np.minimum((y*5).astype(int),4)
    rows=[]
    for channel,v in [('Light',y),('R',x[...,0]),('G',x[...,1]),('B',x[...,2])]:
        rows.append([float(v[band==k].mean()) if np.any(band==k) else np.nan for k in range(4,-1,-1)])
    return pd.DataFrame(rows,index=['Light','R','G','B'],columns=ZONE_NAMES[::-1])
print('SOURCE'); display(channel_table(source).round(3))
print('REFERENCE'); display(channel_table(reference).round(3))
fig,axs=plt.subplots(1,4,figsize=(15,3))
for ax,c,name in zip(axs,range(4),['Light','R','G','B']):
    for region,model in models.items():
        ax.plot(GRID,model['light'] if c==0 else model['rgb'][c-1],label=region)
    ax.plot([0,1],[0,1],'k--',alpha=.3); ax.set_title(name)
    for v in [.2,.4,.6,.8]: ax.axvline(v,color='gray',alpha=.15)
axs[0].legend(); plt.tight_layout(); plt.show(); plt.close(fig)


## Develop controls and local corrections
Defaults are an automatic starting point. `Light` and `Colour` set how strongly the fitted reference transform is applied. The controls below then refine exposure, white-balance bias, contrast, saturation and all five tonal zones. Temperature/tint are relative grading biases, **not camera Kelvin**. Texture and clarity use spatial scales relative to the long edge so preview and export stay consistent.

The optional HSL mixer has eight hue bands; each entry is `[hue shift in degrees, saturation change, lightness change]`. Changes blend smoothly by hue and suppress hue changes near neutral colours. Local controls apply through the inspected masks. These are custom algorithms rather than numerical equivalents of Adobe sliders.


In [ ]:
# 8 · Photographer controls; reusable in the JSON recipe
CONTROLS=dict(strength=.85,tone=.85,color=.80,exposure=0.,temperature=0.,tint=0.,
              contrast=0.,saturation=1.,vibrance=0.,clarity=0.,texture=0.,
              zones=[0.,0.,0.,0.,0.])  # blacks → whites, each -0.2..0.2
LOCAL={k:dict(strength=1.,exposure=0.,saturation=1.) for k in sm}
HSL={name:[0.,0.,0.] for name in ['red','orange','yellow','green','aqua','blue','purple','magenta']}
HUE_CENTERS=np.array([0,30,60,120,180,240,280,320])/360

def hsl_mixer(x):
    from matplotlib.colors import rgb_to_hsv,hsv_to_rgb
    hsv=rgb_to_hsv(np.clip(x,0,1)); h,s,v=np.moveaxis(hsv,-1,0)
    dh=np.zeros_like(h); ds=dh.copy(); dv=dh.copy(); total=dh.copy()
    for center,settings in zip(HUE_CENTERS,HSL.values()):
        distance=np.abs((h-center+.5)%1-.5)
        w=np.exp(-.5*(distance/(24/360))**2)
        dh+=w*settings[0]/360; ds+=w*settings[1]; dv+=w*settings[2]; total+=w
    neutral=np.clip(s/.15,0,1)
    hsv[...,0]=(h+dh/np.maximum(total,1e-6)*neutral)%1
    hsv[...,1]=np.clip(s*(1+ds/np.maximum(total,1e-6)),0,1)
    hsv[...,2]=np.clip(v+dv/np.maximum(total,1e-6)*neutral,0,1)
    return hsv_to_rgb(hsv).astype('float32')
def render(x,masks,controls=None,spatial_edge=None):
    c=CONTROLS if controls is None else controls
    out=np.zeros_like(x)
    for name,mask in masks.items():
        m=models.get(name,models['global']); loc=LOCAL[name]
        graded=apply_model(x,m,c['tone'],c['color'])
        graded=x+loc['strength']*(graded-x)
        graded=srgb_encode(srgb_decode(graded)*2**loc['exposure'])
        gy=light(graded)[...,None]
        graded=gy+(graded-gy)*loc['saturation']
        out+=graded*mask[...,None]
    out=x+c['strength']*(out-x)
    out=srgb_encode(srgb_decode(out)*2**c['exposure'])
    out*=np.array([1+c['temperature']*.15,1+c['tint']*.10,1-c['temperature']*.15],dtype='float32')
    y=light(out)
    out+=(zone_weights(np.clip(y,0,1))@np.asarray(c['zones'],dtype='float32'))[...,None]
    out=(out-.5)*(1+c['contrast'])+.5
    y=light(out); chroma=out-y[...,None]
    sat=np.max(out,-1)-np.min(out,-1)
    out=y[...,None]+chroma*(c['saturation']+c['vibrance']*(1-np.clip(sat,0,1)))[...,None]
    if any(any(v) for v in HSL.values()): out=hsl_mixer(out)
    edge=spatial_edge or max(x.shape[:2])
    y=light(out)
    for key,scale in [('clarity',.008),('texture',.0012)]:
        if c[key]:
            detail=y-gaussian_filter(y,max(.5,edge*scale),mode='reflect',truncate=4)
            out+=c[key]*detail[...,None]
    return out.astype('float32')  # preserve overshoot for diagnostic, clip only at display/export
result=render(source,sm)
show_images([source,np.clip(result,0,1),reference],['Before','Matched','Reference'])


## Automatic comparison: one RAW, three references
Top row: your attached references. Bottom row: the **same source RAW**, independently graded against each reference. The first column shows the unchanged starting development. Fixed controls make this a reference comparison rather than a comparison of hand-selected slider settings.

This comparison uses global landscape grading; no reliable subject correspondence exists across these scenes. Matching does not recreate sun rays, fog or landscape geometry. The results reveal both colour-transfer behaviour and its limits. For local matching, use paired masks on suitable source/reference photos.

`SELECTED_REFERENCE` in the settings cell chooses the look used by the following sliders and high-resolution export. Change it and rerun the notebook to select another. Every run exports **all three preview results and individual recipes**, plus the selected look's 16-bit TIFF. The comparison reflects controls when this cell ran; rerun it after adjusting sliders if you want an updated comparison.


In [ ]:
# 8b · Same RAW × every reference — independently fitted, no chained edits
from copy import deepcopy
COMPARISON_DIR=Path('/content/reference_lab_attached_variations') if Path('/content').exists() else Path('work/attached_variations')
COMPARISON_DIR.mkdir(parents=True,exist_ok=True)
def json_ready(v):
    if isinstance(v,np.ndarray):return v.tolist()
    if isinstance(v,dict):return {k:json_ready(x) for k,x in v.items()}
    if isinstance(v,(list,tuple)):return [json_ready(x) for x in v]
    return v

def compare_references():
    global models
    # Keep source, source masks and controls fixed. Only reference/model changes.
    original_models=models
    original_hash=hashlib.sha256(source.tobytes()).hexdigest()
    records=[]
    try:
        for i,path in enumerate(REFERENCE_PATHS):
            start=time.time()
            ref=load_photo(path,PREVIEW_LONG_EDGE,True)
            masks=region_masks(ref,REFERENCE_MASKS)
            fitted,regions=fit_all(source,ref,sm,masks)
            models=fitted
            graded=render(source,sm)
            name=REFERENCE_ASSETS[i]['name']
            clipped=100*float(np.any((graded<0)|(graded>1),axis=-1).mean())
            rec=dict(name=name,path=path,reference=ref,reference_masks=masks,models=fitted,
                     regions=regions,result=np.clip(graded,0,1),clipped_pct=clipped,
                     seconds=round(time.time()-start,2))
            records.append(rec)
            Image.fromarray(np.uint8(rec['result']*255)).save(COMPARISON_DIR/f'look_{i+1}.jpg',quality=95,
                icc_profile=ImageCms.ImageCmsProfile(ImageCms.createProfile('sRGB')).tobytes())
            recipe={'format':'reference-lab-v1','source_sha256':sha256_file(SOURCE_PATH),
                    'reference_sha256':sha256_file(path),'controls':deepcopy(CONTROLS),
                    'local':deepcopy(LOCAL),'hsl':deepcopy(HSL),'models':fitted,
                    'raw_develop_ev':RAW_DEVELOP_EV,'name':name,
                    'note':'Same source and controls for every look; source masks must be rebuilt for new images.'}
            (COMPARISON_DIR/f'look_{i+1}_recipe.json').write_text(json.dumps(json_ready(recipe),indent=2))
            print(name,':',rec['seconds'],'s;',round(clipped,2),'% out-of-range pixels')
    finally:models=original_models
    assert hashlib.sha256(source.tobytes()).hexdigest()==original_hash,'Source was unexpectedly modified'
    return records

VARIATIONS=compare_references()
cols=len(VARIATIONS)+1
fig,axs=plt.subplots(2,cols,figsize=(4.2*cols,7),facecolor='#111827')
axs=np.asarray(axs).reshape(2,cols)
axs[0,0].imshow(source);axs[0,0].set_title('One source · P1060804.dng' if USE_EMBEDDED_ASSETS else 'One source RAW',color='white')
axs[1,0].imshow(source);axs[1,0].set_title('Unchanged starting development',color='white')
for col,rec in enumerate(VARIATIONS,1):
    axs[0,col].imshow(rec['reference']);axs[0,col].set_title('Reference · '+rec['name'],color='white')
    axs[1,col].imshow(rec['result']);axs[1,col].set_title('Same RAW · look '+str(col),color='white')
for ax in axs.ravel():ax.axis('off')
fig.suptitle('REFERENCE LAB  /  One RAW, different reference looks',color='white',fontsize=18)
fig.tight_layout();fig.savefig(COMPARISON_DIR/'P1060804_three_reference_results.jpg',dpi=150,facecolor=fig.get_facecolor())
plt.show();plt.close(fig)
print('All outputs start from the same decoded RAW; they are not sequential filters.')
# Keep selected look active for sliders, region review and full-resolution export.
active=VARIATIONS[SELECTED_REFERENCE if USE_EMBEDDED_ASSETS else 0]
models=active['models'];valid_regions=active['regions'];reference=active['reference'];rm=active['reference_masks']
REFERENCE_PATH=active['path'];result=render(source,sm)


In [ ]:
# 9 · Live preview sliders; edits update CONTROLS used by export
import ipywidgets as widgets
from IPython.display import display,clear_output
specs={'strength':(0,1,.05),'tone':(0,1,.05),'color':(0,1,.05),'exposure':(-2,2,.05),
       'temperature':(-1,1,.05),'tint':(-1,1,.05),'contrast':(-.5,.5,.025),
       'saturation':(0,2,.05),'vibrance':(-.5,.5,.05),'clarity':(-.5,.5,.05),'texture':(-.5,.5,.05)}
sliders={k:widgets.FloatSlider(value=CONTROLS[k],min=a,max=b,step=d,description=k,
                             continuous_update=False) for k,(a,b,d) in specs.items()}
zone_sliders=[widgets.FloatSlider(value=v,min=-.2,max=.2,step=.005,description=n,
                                 continuous_update=False) for n,v in zip(ZONE_NAMES,CONTROLS['zones'])]
out_widget=widgets.Output()
button=widgets.Button(description='Update visual comparison',button_style='primary')
def update_preview(_=None):
    global result
    CONTROLS.update({k:s.value for k,s in sliders.items()})
    CONTROLS['zones']=[s.value for s in zone_sliders]
    result=render(source,sm)
    with out_widget:
        clear_output(wait=True)
        show_images([source,np.clip(result,0,1),reference],['Before','Matched','Reference'])
button.on_click(update_preview)
display(widgets.HBox([widgets.VBox(list(sliders.values())[:6]),widgets.VBox(list(sliders.values())[6:]),widgets.VBox(zone_sliders)]),button,out_widget)
update_preview()


In [ ]:
# 10 · Regional visual check and clipping diagnostics (rerun after changes)
result=render(source,sm)
for name in valid_regions:
    # Masked images preserve scene layout; gray outside each region.
    images=[source*sm[name][...,None]+.18*(1-sm[name][...,None]),
            np.clip(result,0,1)*sm[name][...,None]+.18*(1-sm[name][...,None]),
            reference*rm[name][...,None]+.18*(1-rm[name][...,None])]
    show_images(images,[name+' before',name+' after',name+' reference'])
rows=[]
for name,mask in {'global':np.ones(source.shape[:2]),**sm}.items():
    selected=mask>.55
    if selected.sum()<64: continue
    rows.append({'region':name,'pixels':int(selected.sum()),
                 'below zero %':100*float(np.any(result[selected]<0,axis=-1).mean()),
                 'above one %':100*float(np.any(result[selected]>1,axis=-1).mean())})
display(pd.DataFrame(rows).round(2))
print('AFTER — 4 channels × 5 tonal zones'); display(channel_table(np.clip(result,0,1)).round(3))
fig,axs=plt.subplots(1,3,figsize=(15,3))
for ax,x,title in zip(axs,[source,np.clip(result,0,1),reference],['Before','After','Reference']):
    for i,col in enumerate(['red','green','blue']):
        ax.hist(x[...,i].ravel()[::8],bins=128,range=(0,1),histtype='step',density=True,color=col)
    ax.set_title(title)
plt.tight_layout(); plt.show(); plt.close(fig)


### Visual approval checklist
1. Compare the **subject first**: skin hue, face brightness, white clothing, specular highlights. Lower colour strength or adjust local saturation if needed.
2. Compare background colour family and brightness; use a corresponding foreground mask when near scenery needs a separate grade.
3. Check light-to-dark transitions and hair edges for halos. A mask that looks plausible at thumbnail size may fail at export size.
4. Check shadows for amplified noise and highlights for clipping. Lower exposure, highlights/whites or matching strength if needed.
5. Judge the overall mood at the same display size, then inspect detail crops. A global histogram cannot establish a visual match.

**Useful local edit:** `LOCAL['subject']['exposure']=0.15` or `LOCAL['background']['saturation']=0.85`, then rerun preview. For HSL, `HSL['green']=[-10,-0.15,0]`. If masks were changed, refit from cell 5 onward. If controls are edited as code, rerun cell 9 to refresh its sliders before clicking its button.


In [ ]:
# 11 · Optional matched-detail inspection — choose independently framed crops
# Normalized x0,y0,x1,y1. Adjust so the crops depict corresponding materials.
SOURCE_CROP=(.25,.25,.75,.75)
REFERENCE_CROP=(.25,.25,.75,.75)
def crop(x,box):
    h,w=x.shape[:2]; a,b,c,d=box
    assert 0<=a<c<=1 and 0<=b<d<=1
    return x[int(b*h):max(int(d*h),int(b*h)+1),int(a*w):max(int(c*w),int(a*w)+1)]
show_images([crop(source,SOURCE_CROP),crop(np.clip(result,0,1),SOURCE_CROP),crop(reference,REFERENCE_CROP)],
            ['Source detail','Graded detail','Reference detail'])


## Export — resolution-aware, tiled grading
Fitting uses preview pixels; export applies the fitted transforms without refitting. Processing uses float32 and row tiles with overlap for clarity/texture filters. Output is **16-bit sRGB TIFF**, an ICC-tagged JPEG preview, masks and a reproducible JSON recipe, bundled into a ZIP. Original RAW remains untouched. The TIFF is a rendered photograph, not a new RAW.

**Memory:** RAW demosaicing is not tiled by LibRaw. A conservative preflight estimates peak RAM; reduce `EXPORT_FULL_RAW`, resolution or use a higher-RAM runtime if it refuses. The output buffer is disk-backed, but RAW decode still needs contiguous RAM. Preview masks are interpolated for export, so fine edges need inspection. Free Colab runtime limits and timings vary. Save downloads or copy the ZIP to Drive before disconnecting.


In [ ]:
# 12 · Full-resolution export and reusable recipe
import tifffile, zipfile, hashlib
OUTPUT_DIR=Path('/content/reference_lab_attached_output') if Path('/content').exists() else Path('work/attached_output')
OUTPUT_DIR.mkdir(exist_ok=True,parents=True)
def file_hash(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''): h.update(block)
    return h.hexdigest()
def serializable(v):
    if isinstance(v,np.ndarray): return v.tolist()
    if isinstance(v,dict): return {k:serializable(x) for k,x in v.items()}
    if isinstance(v,(list,tuple)): return [serializable(x) for x in v]
    return v

def export_photo():
    # Conservative decode estimate, not a guarantee. Inspect sensor dimensions before processing.
    if Path(SOURCE_PATH).suffix.lower() in RAW_EXT:
        import rawpy
        with rawpy.imread(SOURCE_PATH) as raw:
            n=int(raw.sizes.raw_height)*int(raw.sizes.raw_width)
    else:
        with Image.open(SOURCE_PATH) as im: n=im.width*im.height
    need=n*100 + 600*1024**2
    if not EXPORT_FULL_RAW and Path(SOURCE_PATH).suffix.lower() in RAW_EXT: need=need*.45
    available=psutil.virtual_memory().available
    print(f'Conservative decode/export allowance: {need/2**30:.2f} GiB; free: {available/2**30:.2f} GiB')
    if need>available*.8:
        raise MemoryError('Insufficient headroom. Set EXPORT_FULL_RAW=False or use a higher-RAM runtime. Reducing output size alone does not shrink full RAW decoding.')
    full=load_photo(SOURCE_PATH,EXPORT_LONG_EDGE,half=not EXPORT_FULL_RAW)
    h,w=full.shape[:2]; edge=max(h,w)
    mapped={k:resize_float(v,(h,w)) for k,v in sm.items()}
    total=sum(mapped.values())
    mapped={k:v/np.maximum(total,1e-8) for k,v in mapped.items()}
    del total
    temp=OUTPUT_DIR/'pixels.memmap'
    buffer=np.memmap(temp,dtype='uint16',mode='w+',shape=full.shape)
    halo=int(np.ceil(4*max(.5,edge*.008)))+2 if CONTROLS['clarity'] else int(np.ceil(4*max(.5,edge*.0012)))+2 if CONTROLS['texture'] else 0
    clipped=0
    for y in range(0,h,TILE_ROWS):
        end=min(h,y+TILE_ROWS); lo=max(0,y-halo); hi=min(h,end+halo)
        tile=render(full[lo:hi],{k:v[lo:hi] for k,v in mapped.items()},spatial_edge=edge)[y-lo:end-lo]
        clipped+=np.any((tile<0)|(tile>1),axis=-1).sum()
        buffer[y:end]=np.rint(np.clip(tile,0,1)*65535).astype('uint16')
    buffer.flush()
    icc=ImageCms.ImageCmsProfile(ImageCms.createProfile('sRGB')).tobytes()
    tifffile.imwrite(OUTPUT_DIR/'matched_16bit_sRGB.tif',buffer,photometric='rgb',
                     metadata=None,extratags=[(34675,'B',len(icc),icc,False)])
    del buffer; temp.unlink(missing_ok=True)
    preview=render(source,sm)
    Image.fromarray(np.uint8(np.clip(preview,0,1)*255)).save(OUTPUT_DIR/'matched_preview.jpg',quality=95,icc_profile=icc)
    for prefix,masks in [('source',sm),('reference',rm)]:
        for name,mask in masks.items():
            Image.fromarray(np.uint8(np.clip(mask,0,1)*255)).save(OUTPUT_DIR/f'{prefix}_{name}_mask.png')
    recipe={'format':'reference-lab-v1','source':Path(SOURCE_PATH).name,'reference':Path(REFERENCE_PATH).name,
            'source_sha256':file_hash(SOURCE_PATH),'reference_sha256':file_hash(REFERENCE_PATH),
            'controls':CONTROLS,'local':LOCAL,'hsl':HSL,'models':models,
            'raw_develop_ev':RAW_DEVELOP_EV,'raw_preview_half_size':RAW_HALF_SIZE,
            'export_full_raw':EXPORT_FULL_RAW,'output_shape':[h,w],
            'valid_regions':valid_regions,'mask_note':'Saved masks are preview resolution; interpolate to output size.',
            'embedded_assets':USE_EMBEDDED_ASSETS,'selected_reference':SELECTED_REFERENCE,
            'input_assets':{'raw':RAW_ASSET,'references':REFERENCE_ASSETS} if USE_EMBEDDED_ASSETS else None,
            'packages':{p:importlib.metadata.version(p) for p in ['numpy','scipy','Pillow','rawpy','tifffile']}}
    (OUTPUT_DIR/'recipe.json').write_text(json.dumps(serializable(recipe),indent=2))
    (OUTPUT_DIR/'README.txt').write_text('Rendered 16-bit sRGB TIFF; original RAW unchanged. JSON is a custom notebook recipe, not an Adobe preset. Masks are preview resolution. To reuse: load recipe controls/local/hsl/models and rebuild source masks for the new image. Use this same notebook engine. Review every new image visually.')
    zip_path=OUTPUT_DIR.parent/'P1060804_Reference_Lab_Export.zip'
    with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT_DIR.iterdir():
            if p.is_file(): z.write(p,p.name)
        if COMPARISON_DIR.exists():
            for p in COMPARISON_DIR.iterdir():
                if p.is_file():z.write(p,'reference_variations/'+p.name)
    print(f'Exported {w} × {h}. Out-of-range pixels before clipping: {100*clipped/(h*w):.2f}%')
    del full,mapped; gc.collect()
    return zip_path
EXPORT_ZIP=export_photo()
print(EXPORT_ZIP)


In [ ]:
# Kaggle retains exports in the notebook Output tab.
from IPython.display import FileLink, display
display(FileLink(str(EXPORT_ZIP)))


## Reuse a saved recipe (optional)
After loading a new source and its masks, replace the fitted settings below. A recipe transfers a grade; it cannot transfer a segmentation mask to a different composition. Refit against the reference when lighting or content has changed substantially.
```python
saved=json.loads(Path('/content/recipe.json').read_text())
assert saved['format']=='reference-lab-v1'
CONTROLS=saved['controls']; LOCAL=saved['local']; HSL=saved['hsl']
models={k:{p:np.asarray(v,dtype='float32') for p,v in m.items()} for k,m in saved['models'].items()}
RAW_DEVELOP_EV=saved['raw_develop_ev']  # set BEFORE decoding the new source
# Load new source with load_photo, rebuild sm with region_masks, then render.
```

## Implementation references and limits
- [rawpy processing parameters](https://letmaik.github.io/rawpy/api/rawpy.Params.html): linear decode, camera WB and colour space.
- [rembg](https://github.com/danielgatis/rembg): optional salient-object masks, reusable ONNX session. Model download needs internet; disable `AUTO_SUBJECT` and use PNG masks when unavailable.
- [SciPy PCHIP](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.PchipInterpolator.html): monotonic tone curves without interpolation overshoot.

There is no learned perceptual loss or guarantee of matching scene lighting. This design uses region-conditioned statistical fitting plus visual human review, with editable curves, colour and local controls. It deliberately does not hallucinate texture. Denoising, lens correction, optical blur, film grain, calibrated camera profiles and Adobe XMP export are not implemented. Strong clipping, coloured stage lights, monochrome references and mismatched subjects may need manual grading or a different reference.

**Validation scope:** embedded-file checksum verification, your P1060804 DNG decode, three independent reference fits, comparison images and TIFF export were executed during this revision. Optional automatic masks and the live Colab UI still need checking in Colab. Use a fresh runtime if pip reports incompatible preinstalled packages.
